In [ ]:

# No terminal, uma vez: python -m pip install xlrd pandas
import pandas as pd



In [ ]:
# a tabela fica na mesma pasta do notebook
path = "Tabela 1.1.1.xls"

In [ ]:
indicador_1 = pd.read_excel(path, engine="xlrd")

In [ ]:
indicador_1


O `.xls` do IBGE mistura **título**, cabeçalho em várias linhas e **notas no rodapé**.  
Cortamos isso e deixamos só: **nomes das colunas + números**.

Tabela 1.1.1 - Número médio de horas semanais dedicadas aos cuidados de pessoas e/ou afazeres domésticos das pessoas de 14 anos ou mais de idade, na semana de referência, por sexo e cor ou raça, segundo Grandes Regiões e Unidades da Federação - Brasil - 2022	

In [ ]:
# Lê sem usar a 1ª linha como nome de coluna (senão o título vira "coluna")
bruto = pd.read_excel(path, engine="xlrd", header=None)

# Linhas 0–7: título/cabeçalho  |  8–40: resultados  |  41+: fonte e notas
indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)

# horas são número, não texto
for col in indicador_1.columns[1:]:
    indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")

indicador_1


In [ ]:
# Média nacional = linha Brasil (dado do IBGE), não a média das linhas da tabela
media_nacional = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil", "total"].item()
media_nacional

In [ ]:
media_nacional

Qual(is) estado(s) está acima da média nacional de afazeres domésticos?

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

acima_media = estados.loc[
    estados["total"] > media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados acima da média: {len(acima_media)}")
acima_media

Quais estados estão abaixo da média?

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

abaixo_media = estados.loc[
    estados["total"] < media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados abaixo da média: {len(abaixo_media)}")
abaixo_media

Qual é o percentual de diferença no Brasil que mulheres trabalham mais que homens em casa?

In [ ]:

brasil = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil"].iloc[0]

homem = (brasil["homem_branca"] + brasil["homem_preta_parda"]) / 2
mulher = (brasil["mulher_branca"] + brasil["mulher_preta_parda"]) / 2

# quanto % a mais as mulheres trabalham em relação aos homens
percentual = (mulher - homem) / homem * 100

print(f"Homens (média das cores):   {homem:.2f} h/semana")
print(f"Mulheres (média das cores): {mulher:.2f} h/semana")
print(f"Mulheres trabalham {percentual:.1f}% a mais que homens em casa, no Brasil.")

# recorte por cor (a tabela traz isso de fato)
perc_branca = (brasil["mulher_branca"] / brasil["homem_branca"] - 1) * 100
perc_preta_parda = (brasil["mulher_preta_parda"] / brasil["homem_preta_parda"] - 1) * 100
print(f"Branca:        mulheres +{perc_branca:.1f}%")
print(f"Preta ou parda: mulheres +{perc_preta_parda:.1f}%")


Agora responda:
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas em Minas Gerais?
- Quantas horas mulheres pretas trabalham a mais que homens brancos no Brasil?
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas no Amapá?
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas no Paraná?
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas por região?


In [ ]:
def linha(local):
    return indicador_1.loc[indicador_1["uf_regiao"] == local].iloc[0]

# Minas Gerais: mulheres pretas/pardas x mulheres brancas
mg = linha("Minas Gerais")
dif_mg = mg["mulher_preta_parda"] - mg["mulher_branca"]
print(f"Minas Gerais: mulheres pretas ou pardas trabalham {dif_mg:.1f} h a mais que mulheres brancas")

In [ ]:
# Brasil: mulheres pretas/pardas x homens brancos
br = linha("Brasil")
dif_br = br["mulher_preta_parda"] - br["homem_branca"]
print(f"Brasil: mulheres pretas ou pardas trabalham {dif_br:.1f} h a mais que homens brancos")

In [ ]:
# Amapá: mulheres pretas/pardas x mulheres brancas
ap = linha("Amapá")
dif_ap = ap["mulher_preta_parda"] - ap["mulher_branca"]
print(f"Amapá: diferença de {dif_ap:.1f} h (valor negativo = mulheres pretas ou pardas trabalham MENOS que as brancas)")

In [ ]:
# Paraná: mulheres pretas/pardas x mulheres brancas
pr = linha("Paraná")
dif_pr = pr["mulher_preta_parda"] - pr["mulher_branca"]
print(f"Paraná: mulheres pretas ou pardas trabalham {dif_pr:.1f} h a mais que mulheres brancas")

In [ ]:
# Por região: mulheres pretas/pardas x mulheres brancas
por_regiao = indicador_1[indicador_1["uf_regiao"].isin(regioes)][["uf_regiao", "mulher_branca", "mulher_preta_parda"]].copy()
por_regiao["diferenca_horas"] = (por_regiao["mulher_preta_parda"] - por_regiao["mulher_branca"]).round(1)
por_regiao.sort_values("diferenca_horas", ascending=False)

## Respostas

- **Minas Gerais:** mulheres pretas ou pardas trabalham **0,9 h** a mais por semana que mulheres brancas (23,1 h x 22,2 h).
- **Brasil:** mulheres pretas ou pardas trabalham **10,3 h** a mais que homens brancos (22,0 h x 11,7 h), quase o dobro.
- **Amapá:** é a exceção: mulheres pretas ou pardas trabalham **0,7 h a menos** que mulheres brancas (15,9 h x 16,6 h).
- **Paraná:** mulheres pretas ou pardas trabalham **0,5 h** a mais que mulheres brancas (19,4 h x 18,9 h).
- **Por região:** Sudeste +1,3 h, Norte +1,2 h, Centro-Oeste +1,1 h, Nordeste +1,0 h e Sul +0,3 h.